In [3]:
!pip install ultralytics
!pip install opencv-python
!pip install mediapipe
!pip install wandb
!pip install scikit-learn

In [1]:
import wandb
wandb.login()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /Users/amber/.netrc.
wandb: Currently logged in as: amberchee (amberchee-xiamen-university-malaysia) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split

In [3]:
wandb.init(
    project="hazard_classification",
    name="efficientnet_v1"
)

In [4]:
DATASET_PATH = "./hazard_dataset"
batch_size = 32
epochs = 30
lr = 0.001

In [5]:
# Dataset Preprocessing

def get_transformed():
  train_transformed = transforms.Compose([
      transforms.Resize((224, 224)),
      transforms.RandomHorizontalFlip(p=0.5),
      transforms.RandomRotation(15),
      
      transforms.ColorJitter(
          brightness=0.2,
          contrast=0.2,
          saturation=0.2,
          hue=0.05
      ),

      transforms.ToTensor(),

      transforms.Normalize(
          mean=[0.485, 0.456, 0.406],
          std=[0.229, 0.224, 0.225]
      ) 
  ])
  
  val_transformed = transforms.Compose([
      transforms.Resize((224, 224)),
      transforms.ToTensor(),
      transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
  ])

  return train_transformed, val_transformed

In [6]:
# Load & Split Dataset

def load_datasets(data_path):
  train_t, val_t = get_transformed()

  full_dataset = datasets.ImageFolder(data_path, transform=train_t)
  class_labels = full_dataset.classes

  print("Classes: ", class_labels)
  print("Total Images: ", len(full_dataset))

  total = len(full_dataset)
  train_size = int(0.8 * total)
  val_size = int(0.1 * total)
  test_size = total - train_size - val_size

  train_ds, val_ds, test_ds = random_split(full_dataset, [train_size, val_size, test_size])

  val_ds.dataset.transform = val_t
  test_ds.dataset.transform = val_t

  train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
  val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
  test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)

  return train_loader, val_loader, test_loader, class_labels


In [7]:
def build_efficientnet(num_classes, device):
  model = models.efficientnet_b0(pretrained=True)
  in_features = model.classifier[1].in_features
  model.classifier[1] = nn.Linear(in_features, num_classes) # Reduce output classes to 4
  return model

In [8]:
def train_model(model, train_loader, val_loader, device):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    for epoch in range(epochs):
        model.train()
        train_correct = 0
        train_total = 0
        train_loss = 0

        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

            _, preds = outputs.max(1)
            train_total += labels.size(0)
            train_correct += preds.eq(labels).sum().item()

        train_acc = train_correct / train_total
        train_loss = train_loss / len(train_loader)

        # validation
        model.eval()
        val_correct = 0
        val_total = 0
        val_loss = 0

        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                outputs = model(imgs)
                _, preds = outputs.max(1)

                val_total += labels.size(0)
                val_correct += preds.eq(labels).sum().item()

        val_acc = val_correct / val_total
        val_loss = val_loss / len(val_loader)

        print(f"Epoch {epoch+1}/{epochs} | Train Acc: {train_acc:.4f} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

        wandb.log({
            "epoch": epoch+1,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "train_acc": train_acc,
            "val_acc": val_acc
        })

        

In [9]:
# Model Evaluation 

def evaluate_model(model, test_loader, device):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for imgs, labels in test_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            _, preds = outputs.max(1)

            total += labels.size(0)
            correct += preds.eq(labels).sum().item()

    acc = 100 * correct / total
    print("Final Test Accuracy:", acc)
    return acc


In [10]:
def main():
    wandb.init(project="hazard-classifier")
    
    if torch.backends.mps.is_available():
      device = torch.device("mps")
      print("Using Apple MPS GPU")

    elif torch.cuda.is_available():
      device = torch.device("cuda")
      print("Using CUDA GPU")

    else:
      device = torch.device("cpu")
      print("Using CPU")

    train_loader, val_loader, test_loader, class_names = load_datasets(DATASET_PATH)

    model = build_efficientnet(len(class_names), device)
    model = model.to(device)
    model = model.float()

    train_model(model, train_loader, val_loader, device)

    evaluate_model(model, test_loader, device)

    torch.save(model.state_dict(), "hazard_model.pth")
    wandb.save("hazard_model.pth")

if __name__ == "__main__":
    main()

Using Apple MPS GPU
Classes:  ['fire', 'flood', 'landslide', 'normal']
Total Images:  2110


/Users/amber/miniconda3/envs/tf310/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/Users/amber/miniconda3/envs/tf310/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=EfficientNet_B0_Weights.IMAGENET1K_V1`. You can also use `weights=EfficientNet_B0_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Epoch 1/30 | Train Acc: 0.8993 | Train Loss: 0.3400 | Val Loss: 0.0000 | Val Acc: 0.9526
Epoch 2/30 | Train Acc: 0.9615 | Train Loss: 0.1245 | Val Loss: 0.0000 | Val Acc: 0.9716
Epoch 3/30 | Train Acc: 0.9757 | Train Loss: 0.0818 | Val Loss: 0.0000 | Val Acc: 0.9526
Epoch 4/30 | Train Acc: 0.9650 | Train Loss: 0.1025 | Val Loss: 0.0000 | Val Acc: 0.9716


KeyboardInterrupt: 